# CUAD class-imbalance experiment (task #12)

Select a **T4 GPU runtime**, then run the cells in order. All the work is in
`scripts/imbalance_experiment.py`; see `docs/class-imbalance.md` for the design and decision rule.

Four arms (`bce`, `pos_weight`, `focal`, `downsample`) on five contract folds: 20 runs, about
13 hours at most. Finished runs are saved to Google Drive and skipped, so **after a disconnect,
just run everything again**. To share the work, each person sets `FOLDS` to different folds.

The official test split is never used.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Break-Through-Tech/Accenture-1T-contract-review-challenge.git"
REPO_REF = "task-12-class-imbalance"
REPO_DIR = Path("/content/contract-review")
DRIVE_DIR = Path("/content/drive/MyDrive/contract-review/imbalance")  # survives disconnects

FOLDS = [0, 1, 2, 3, 4]  # when sharing the work, each person lists different folds
ARMS = ["bce", "pos_weight", "focal", "downsample"]
SEED = 42
EPOCHS = 2
BATCH_SIZE = 8          # lower to 4 if the GPU runs out of memory; use one value for all runs
LEARNING_RATE = 2e-5
SMOKE_TEST = False      # True: a few steps per arm, to check setup. Saved apart and never compared
RUNS_DIR = DRIVE_DIR / ("smoke" if SMOKE_TEST else "runs")

## 1. Save location, checkout and locked environment
Colab asks for permission to use Drive. An existing checkout is used as-is (no pull).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
required = [
    "pyproject.toml", "uv.lock", "src/imbalance.py", "scripts/imbalance_experiment.py",
    "data/train_separate_questions.json", "docs/tuned-baseline-results/folds.csv",
]
missing = [name for name in required if not (REPO_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Checkout is missing {missing}. Push the branch first.")
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
UV = [sys.executable, "-m", "uv"]
subprocess.run(UV + ["sync", "--locked", "--dev", "--python", "3.12"], cwd=REPO_DIR, check=True)

def run_project(*args):
    # Colab does not show a child process's output unless it is relayed line by line.
    process = subprocess.Popen(
        UV + ["run", "--locked", "--no-sync", *map(str, args)], cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args)

## 2. GPU and software checks
Stop here if either check fails.

In [ ]:
run_project("python", "-c", '''
import torch, transformers
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
print("Transformers:", transformers.__version__)
assert torch.cuda.is_available(), "No usable CUDA device in the project environment"
print("GPU:", torch.cuda.get_device_name(0),
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
''')
run_project("python", "-B", "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_imbalance.py", "tests/test_loss.py")

## 3. Re-chunk and label the training split with legal-bert's tokenizer
About two minutes, repeated each session because it is written to the runtime's disk. Expect 14,023
chunks and `longest_chunk_tokens` of 512. The result is identical each time, which `summarize`
checks using a hash, so runs from different sessions and accounts can be combined.

In [ ]:
run_project("python", "-B", "-m", "scripts.imbalance_experiment", "prepare")

## 4. Train
Each arm finishes a fold before the next fold starts, so stopping early still leaves a fair
comparison on the folds that all arms finished. A failed run is reported and the loop continues.
**Check `seconds_per_chunk` after the first run** to estimate the total time. `amp_skipped_steps`
should be small.

In [ ]:
failed = []
smoke = ["--train-contracts", 6, "--validation-contracts", 2, "--max-steps", 4] if SMOKE_TEST else []
for fold in FOLDS:
    for arm in ARMS:
        print(f"===== {arm} fold={fold} =====", flush=True)
        try:
            run_project(
                "python", "-B", "-m", "scripts.imbalance_experiment", "train",
                "--arm", arm, "--validation-fold", fold, "--output-dir", RUNS_DIR,
                "--seed", SEED, "--epochs", EPOCHS, "--batch-size", BATCH_SIZE,
                "--learning-rate", LEARNING_RATE, *smoke,
            )
        except subprocess.CalledProcessError as exc:
            failed.append((arm, fold))
            print(f"FAILED: {arm} fold={fold} ({exc}). See the traceback above.")
print("Failed runs:", failed or "none")

## 5. Compare and download
Needs at least two folds finished by every arm. If teammates ran other folds, first copy their
`runs` folders into your Drive and add them to `--runs-dir`. Rerun this after more folds finish.
Commit the downloaded files to `docs/class-imbalance-results/`.

In [ ]:
import json
import pandas as pd

SUMMARY_DIR = DRIVE_DIR / "summary"
run_project(
    "python", "-B", "-m", "scripts.imbalance_experiment", "summarize",
    "--runs-dir", RUNS_DIR, "--output-dir", SUMMARY_DIR, "--seed", SEED,
)
metadata = json.loads((SUMMARY_DIR / "metadata.json").read_text())
print("Folds compared:", metadata["folds_compared"], "of", metadata["folds_expected"])
print(json.dumps(metadata["decision"], indent=2))
display(pd.read_csv(SUMMARY_DIR / "arm_summary.csv"))
display(pd.read_csv(SUMMARY_DIR / "pairwise_bootstrap.csv"))
per_category = pd.read_csv(SUMMARY_DIR / "per_category.csv")
display(per_category.pivot(index="category", columns="arm", values="f1"))

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/class-imbalance-results", "zip", root_dir=SUMMARY_DIR)
files.download(archive)